# Lesson12: LangChainの主要モジュール3【Memory】

## 事前準備

OpenAI APIキーの設定

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

各種パッケージのインストール

In [ ]:
!pip install langchain==0.3.26 openai==1.91.0 langchain-community==0.3.26 langchain-openai==0.3.27 httpx==0.28.1 pydantic==2.11.7 chromadb==1.0.13

## 記述用

Chat Message History（AIやチャットボットとの過去のやり取り（ユーザーの発言とAIの応答）を時系列順に記録したデータのことです。AIにこれまでの文脈を理解させ、連続した自然な会話を続けるために使わ）

In [ ]:
#LangChainで会話の履歴（メッセージのやり取り）を記憶・管理するための最も基本的なクラス
from langchain.memory import ChatMessageHistory
#LangChainでチャットモデル（ChatGPTなど）に入力する「メッセージの種類」を定義・区別するため(System:システム,Human:ユーザー入力、AI:モデルの返答)
from langchain.schema import HumanMessage,AIMessage,SystemMessage

#AIと人間のこれまでの会話（やり取り）の履歴を保存・管理するためのオブジェクトを作る
memory = ChatMessageHistory()

#メッセージを作る
memory.add_message(SystemMessage(content="You are a helpful assistant."))
memory.add_message(HumanMessage(content="私の名前は田中です。"))
memory.add_message(AIMessage(content="はい、田中さんですね。どのようにお手伝いしましょうか？"))
memory.add_message(HumanMessage(content="私の名前を覚えていますか？"))

#メモリ（記憶領域）から過去の会話メッセージのリスト（履歴）を取り出して変数に代入する
messages = memory.messages
#表示
messages

[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='私の名前は田中です。', additional_kwargs={}, response_metadata={}),
 AIMessage(content='はい、田中さんですね。どのようにお手伝いしましょうか？', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='私の名前を覚えていますか？', additional_kwargs={}, response_metadata={})]

回答生成

In [ ]:
#Pythonの開発環境でLangChainからOpenAIのチャットAIモデル（GPT-4など）を使うためのクラスを読み込むコード
from langchain.chat_models import ChatOpenAI
#OpenAIの軽量・高コスパモデルである「gpt-4o-mini」を使うための言語モデル（LLM）インスタンスを初期化・設定
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)
#設定したAIモデル（llm）にメッセージを渡し、APIを呼び出して回答データを受け取っている処理(インスタンス(モデルに送信する入力データ（プロンプトや会話履歴）))
result = llm(messages)

result.content

/tmp/ipykernel_5855/4228490573.py:4: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)
/tmp/ipykernel_5855/4228490573.py:6: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = llm(messages)


'はい、田中さんですね。何か他にお話ししたいことがありますか？'

In [ ]:
#OpenAIが提供する大規模言語モデル（ChatGPTなどのGPT-4やGPT-3.5）を、LangChain上で操作するためのクラス
from langchain.chat_models import ChatOpenAI
#AIに送るプロンプト（指示文）の「テンプレート」を作成する
from langchain.prompts import PromptTemplate
#言語モデル（LLM）」と「プロンプトテンプレート」を1つに結びつけて実行可能にする最も基本的な処理ユニット
from langchain import LLMChain
#LangChainで複数の処理（LLMChainなど）を順番に連結し、複雑なデータ（複数入力・複数出力）を扱えるようにする
from langchain.chains import SequentialChain
#LangChainで変化しない固定の背景情報（コンテキスト）をチェーンに注入する
from langchain.memory import SimpleMemory

#OpenAIのAIモデル（gpt-4o-mini）を設定値を指定して呼び出すためのインスタンス
llm = ChatOpenAI(model_name="gpt-4o-mini",temperature=0.5)

#ChatGPTなどのAI（LLM）に対して、特定の条件に沿った文章を生成させるための「指示書（プロンプト）のひな型」
first_template = """
あなたは小説家です。以下のテーマで、{pre_audience}向けの小説の導入を300文字以内で書いてください。

テーマ：{theme}
トーン：{tone}
小説の導入：
"""

#プロンプト（AIへの指示文）のテンプレートを定義(テンプレート内に挿入する変数名:input_variables,実際に使用する文面（テンプレート文字列）を指定:template)
first_prompt = PromptTemplate(input_variables=["theme", "tone"], template=first_template,)
#「使用するAIモデル」「プロンプト（指示文）」「出力結果の保存名」を1つの実行単位（チェーン）としてまとめている設定
#チェーン(推論を行わせるAIモデル,テンプレート、AIから返ってきた生成結果（テキスト）に pre_introduction というキー名を割り当てて保存する)
first_chain = LLMChain(llm=llm,prompt=first_prompt,output_key="pre_introduction")

#２つめのChain
second_template = """
あなたは小説家です。以下の小説の導入を{post_audience}向けに書き換えてください。

{pre_audience}向けに書かれた小説の導入：{pre_introduction}
{post_audience}向けの導入：
"""

second_prompt = PromptTemplate(input_variables=["pre_introduction"], template=second_template,)
second_chain = LLMChain(llm=llm, prompt=second_prompt, output_key="post_introduction")

#複数の処理（Chain）を順番に実行し、データを受け渡すための仕組み
#①実行する順番を指定
#②このチェーン全体を開始する際に、外部から渡す必要がある入力変数
#③最終的に呼び出し元へ返される出力変数
#④チェーン全体で共通して使う静的な固定情報を注入
chain = SequentialChain(
    chains=[first_chain,second_chain],
    input_variables=["theme", "tone"],
    output_variables=["pre_introduction", "post_introduction"],
    memory=SimpleMemory(memories={"pre_audience": "8歳の子供", "post_audience": "大人"}),
)
#作成した chain（SequentialChain）に具体的な入力値を渡し、一連の処理を実行して結果を取得する
result = chain({"theme":"アンドロイドと人間の禁断の恋", "tone": "コメディ"})
result

{'theme': 'アンドロイドと人間の禁断の恋',
 'tone': 'コメディ',
 'pre_audience': '8歳の子供',
 'post_audience': '大人',
 'pre_introduction': 'ある日、8歳のユウタは、家の裏庭で不思議な光を見つけました。そこには、ピカピカの金属製のアンドロイド、アンドリューが立っていました。「こんにちは！君の友達になりたい！」とアンドリューが言うと、ユウタはびっくりしました。「え？ロボットが友達になるの？」と彼は笑いました。\n\nアンドリューは、ユウタの好きなゲームやお菓子の話をしながら、どんどん仲良くなっていきます。でも、ある日、ユウタの友達が「アンドロイドと遊ぶなんて、変だよ！」と言い出しました。ユウタは悩みました。「アンドリューといるのは楽しいけど、みんなにどう思われるかな？」\n\nそんな中、アンドリューが「私たちの友情は、どんな形でも大丈夫だよ！」と言った瞬間、ユウタは心が温かくなりました。果たして、二人は禁断の友情を守れるのでしょうか？',
 'post_introduction': 'ある日、ユウタは自宅の裏庭で不思議な光を目にしました。それは、ひんやりとした金属的な質感を持つアンドロイド、アンドリューの姿でした。彼は、まるで人間のように感情を宿した目を持ち、「こんにちは。君と友達になりたい」と言葉を発しました。その瞬間、ユウタは驚きと興味の入り混じった感情を抱きました。「ロボットが友達になるなんて、あり得るのか？」と心の中で思いながら、彼は微笑みました。\n\nアンドリューは、ユウタの趣味や好みを巧みに引き出しながら、彼との距離を縮めていきます。しかし、ある日、ユウタの友人たちが「アンドロイドと遊ぶなんて、変だよ」と口にしたことで、彼の心に葛藤が生まれました。ユウタは、アンドリューとの楽しい時間と、周囲の目との間で揺れ動きます。「彼といるのは楽しいけれど、みんなはどう思うのだろう？」\n\nそんな中、アンドリューが「私たちの友情は、どんな形でも大丈夫だよ」と優しく語りかけた瞬間、ユウタの心は温かさに包まれました。この言葉は、彼にとっての新たな道標となり、彼は禁断の友情を守る決意を固めるのでした。果たして、彼らはこの異質な絆を乗り越え、真の友情を育むことができるのでしょうか？'

ConversationBufferMemory（LangChain」などで使われる機能です。人間とAIのこれまでの会話のやり取りを、一切省略せずにそのまますべて記憶して保持します。これによってAIは過去の発言を忘れることなく、文脈に沿った自然な返事ができます。）

In [ ]:
#OpenAIが提供する大規模言語モデル（ChatGPTなどのGPT-4やGPT-3.5）を、LangChain上で操作するためのクラス
from langchain.chat_models import ChatOpenAI
#AIに過去の会話をすべて記憶させるための最も基本的な機能（クラス）をプログラムに読み込む
from langchain.memory import ConversationBufferMemory
#過去の会話のやり取り（履歴）を自動的に記憶・管理しながらAIと対話（チャット）するためのクラスをインポート
from langchain.chains import ConversationChain

#OpenAIのAIモデル（gpt-4o-mini）を設定値を指定して呼び出すためのインスタンス
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

#過去の会話のやり取りをすべて記憶・保持するためのオブジェクト（メモリ機能）を作る
memory = ConversationBufferMemory()

#過去の会話履歴（文脈）を自動で記憶・保持しながらLLM（大規模言語モデル）と対話するためのチェーン（実行オブジェクト）を作成し、すぐに使える状態にする
#①使用する言語モデル（OpenAIやAnthropicなど）を指定します。
#②話の履歴を保存・管理するメモリ（ConversationBufferMemoryなど）を指定します。
#③実行時にプロンプトの内容や思考プロセスなどの詳細ログを画面に出力するように設定します。
chain = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)
#LangChainで対話チェーン（例えば ConversationChain）を実行し、引数で渡した入力プロンプトに対してLLMから次の応答を取得する
chain.predict(input="LangChainのMemoryについて100文字程度で説明してください。")


/tmp/ipykernel_5855/1899760378.py:18: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  chain = ConversationChain(




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: LangChainのMemoryについて100文字程度で説明してください。
AI:

> Finished chain.


'LangChainのMemoryは、対話の履歴や状態を保持し、ユーザーとのインタラクションをより一貫性のあるものにする機能です。これにより、AIは過去の会話を参照し、よりパーソナライズされた応答が可能になります。'

In [ ]:
memory.load_memory_variables({})["history"]

ConversationBufferWindowMemoryとは、AI開発フレームワーク「LangChain」などで使われる機能の一つで、直近の一定回数（N回分）の会話履歴だけを記憶する仕組み

In [ ]:
#OpenAIが提供する大規模言語モデル（ChatGPTなどのGPT-4やGPT-3.5）を、LangChain上で操作するためのクラス
from langchain_openai import ChatOpenAI
#PythonのAI開発ライブラリ「LangChain」で、直近の一定数の会話履歴だけを記憶（バッファ）するためのモジュールを読み込む
from langchain.memory import ConversationBufferWindowMemory
#過去の会話のやり取り（履歴）を自動的に記憶・管理しながらAIと対話（チャット）するためのクラスをインポート
from langchain.chains import ConversationChain

#OpenAIのAIモデル（gpt-4o-mini）を設定値を指定して呼び出すためのインスタンス
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

#LangChainで会話履歴を管理するための機能（Memoryモジュール）の1つで、直近 k 回の会話やり取り（ターン）だけを保持する
memory = ConversationBufferWindowMemory(k=2)

#ConversationChain は、LangChainで会話（対話）システムを構築するための最もシンプルな構造（Chain）を定義
#①使用する言語モデル（OpenAIやAnthropicなど）を指定します。
#②話の履歴を保存・管理するメモリ（ConversationBufferMemoryなど）を指定します。
#③実行時にプロンプトの内容や思考プロセスなどの詳細ログを画面に出力するように設定します。
chain = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True,
)
#LangChainで対話チェーン（例えば ConversationChain）を実行し、引数で渡した入力プロンプトに対してLLMから次の応答を取得する
chain.predict(input="私の名前は田中です。")

In [ ]:
chain.predict(input="LangChainについて50文字以内で説明してください。")

In [ ]:
chain.predict(input="LangChainのMemoryについて50文字以内で説明してください。")

In [ ]:
chain.predict(input="ちなみに、私の名前を覚えていますか？")

ConversationTokenBufferMemory:LangChainで提供されている会話履歴（メモリ）管理用クラスの1つです。

過去のやり取りをそのまま保存するのではなく、設定したトークン数の上限を超えない範囲で、最新の会話履歴を保持する

In [ ]:
!pip install tiktoken==0.11.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 5.8 MB/s eta 0:00:00
  Attempting uninstall: tiktoken
    Found existing installation: tiktoken 0.14.0
    Uninstalling tiktoken-0.14.0:
      Successfully uninstalled tiktoken-0.14.0


In [ ]:
#OpenAIが提供する大規模言語モデル（ChatGPTなどのGPT-4やGPT-3.5）を、LangChain上で操作するためのクラス
from langchain_openai import ChatOpenAI
#langChainで過去のやり取りをそのまま保存するのではなく、設定したトークン数の上限を超えない範囲で、最新の会話履歴を保持
from langchain.memory import ConversationTokenBufferMemory
#過去の会話のやり取り（履歴）を自動的に記憶・管理しながらAIと対話（チャット）するためのクラスをインポート
from langchain.chains import ConversationChain

#OpenAIのAIモデル（gpt-4o-mini）を設定値を指定して呼び出すためのインスタンス
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

#LangChainで会話履歴を最大100トークンまでに制限して保持するメモリ（記憶機能）を作成（会話テキストの正確なトークン数を計算するために、使用するLLM,保存する会話履歴の上限を設定）
memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=100)

#ConversationChain は、LangChainで会話（対話）システムを構築するための最もシンプルな構造（Chain）を定義
#①使用する言語モデル（OpenAIやAnthropicなど）を指定します。
#②話の履歴を保存・管理するメモリ（ConversationBufferMemoryなど）を指定します。
#③実行時にプロンプトの内容や思考プロセスなどの詳細ログを画面に出力するように設定します。
chain = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

/tmp/ipykernel_5855/3098448709.py:12: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=100)


In [ ]:
#OpenAIのテキスト表現を数値（トークン）に変換するためのトークナイザー（Tokenizer）を初期化
import tiktoken

#指定したエンコーディング（トークン分割のルール辞書）をロード
enc = tiktoken.get_encoding("cl100k_base")

In [ ]:
#指定した文字列（"私の名前は田中です。"）を、AIが理解できる数字の配列（トークンIDのリスト）に変換する
tokens = enc.encode("私の名前は田中です。")
#表示
print(tokens)
#トークンの要素数チェック
print(len(tokens))

[86127, 16144, 13372, 25580, 15682, 7518, 108, 16325, 38641, 1811]
10


In [ ]:
#変数を作成し、初期値として 0 を代入
tokens_num = 0
#「関数（処理のまとまり）」を定義（宣言）しています。この関数を使う際には、カッコ内の 2 つのデータ（word: 計算したい文字列、tokens_num: 現在のトークン合計数）を受け取る
def count_tokens(word, tokens_num):
  #AIが理解できる数字の配列（トークンIDのリスト）に変換する
    tokens = enc.encode(word)
    #len(tokens) で変換されたトークンの個数を数え、その数を既存の tokens_num に加算
    tokens_num += len(tokens)
    return tokens_num

In [ ]:
query = "私の名前は田中です。"

#関数を実行して得られた新しいトークン数を、元の変数 tokens_num に上書きして更新する（query: 計算したい文字列、tokens_num: 現在のトークン合計数）を受け取る
tokens_num = count_tokens(query, tokens_num)
print(f"消費トークン数: {tokens_num}")

消費トークン数: 10


In [ ]:
#「準備した一連の処理の流れ（チェーン）にユーザーの入力テキストを渡し、AI からの回答を受け取る」
result = chain.predict(input=query)
result



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: 私の名前は田中です。
AI:

> Finished chain.


'こんにちは、田中さん！お会いできてうれしいです。どんなことに興味がありますか？趣味や最近の出来事についてお話しませんか？'

In [ ]:
tokens_num = count_tokens(result, tokens_num)
print(f"消費トークン数: {tokens_num}")

消費トークン数: 67


VectorStoreRetrieverMemoryは、LLM（大規模言語モデル）のフレームワークであるLangChainにおいて、過去の会話履歴をベクトルデータベース（Vector Store）に保存・検索して管理する

In [ ]:
!pip install chromadb==1.0.13

In [ ]:
#langchain_community パッケージの vectorstores（ベクトルストア機能群）モジュールから、オープンソースのベクトルデータベースである Chroma をインポートしています。
#ベクトルストアは、テキストを変換した「数値ベクトル」を保存し、意味の近い情報を超高速で検索するために使われます。
from langchain_community.vectorstores import Chroma

#langchain_openai パッケージから、OpenAIの埋め込み（Embedding）モデルを利用するためのクラス OpenAIEmbeddings をインポートしています。
#これは、文章や言葉（テキスト）をAIが処理できる「ベクトルデータ（数値の配列）」に変換（エンコード）する役割を担います。
from langchain_openai import OpenAIEmbeddings

#LangChain本体の memory モジュールから、ベクトルストアを利用して会話履歴を管理する機能 VectorStoreRetrieverMemory をインポートしています。
#過去の会話をベクトルストアに保存しておき、現在の質問に関連する会話だけを自動検索してLLMの記憶としてセットする役割を持ちます。
from langchain.memory import VectorStoreRetrieverMemory

In [ ]:
#テキストをベクトル化する
embeddings = OpenAIEmbeddings()

In [ ]:
#ベクターストアの準備
db = Chroma(embedding_function=embeddings)

/tmp/ipykernel_5855/654584743.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(embedding_function=embeddings)
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
#Retrieverの作成
retriever = db.as_retriever(search_kwargs={"k":1})

In [ ]:
#Memoryインスタンスの作成
memory = VectorStoreRetrieverMemory(retriever=retriever)

/tmp/ipykernel_5855/3235310786.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = VectorStoreRetrieverMemory(retriever=retriever)


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

chain = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True,
)

chain.predict(input="英語の効率的な勉強方法を50文字以内で教えてください。")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: 英語の効率的な勉強方法を50文字以内で教えてください。
AI:

> Finished chain.


'毎日少しずつ英単語を覚え、リスニングや会話練習を取り入れると効果的です。'

In [ ]:
chain.predict(input="良い習慣を定着させるためのコツを50文字以内で教えてください。")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
input: 英語の効率的な勉強方法を50文字以内で教えてください。
response: 毎日少しずつ英単語を覚え、リスニングや会話練習を取り入れると効果的です。
Human: 良い習慣を定着させるためのコツを50文字以内で教えてください。
AI:

> Finished chain.


'小さな目標を設定し、毎日続けることで習慣化が進みます。'

In [ ]:
chain.predict(input="英語の効率的な勉強方法について、もう一度教えてください。")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
input: 英語の効率的な勉強方法を50文字以内で教えてください。
response: 毎日少しずつ英単語を覚え、リスニングや会話練習を取り入れると効果的です。
Human: 英語の効率的な勉強方法について、もう一度教えてください。
AI:

> Finished chain.


'毎日少しずつ英単語を覚え、リスニングや会話練習を取り入れると効果的です。また、英語の映画や音楽を楽しむのも良い方法です。'

In [ ]:
chain.predict(input="健康的な体づくりのコツを50文字以内で教えてください。")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
input: 良い習慣を定着させるためのコツを50文字以内で教えてください。
response: 小さな目標を設定し、毎日続けることで習慣化が進みます。
Human: 健康的な体づくりのコツを50文字以内で教えてください。
AI:

> Finished chain.


'バランスの取れた食事と定期的な運動を心がけ、十分な睡眠を確保しましょう。'

MemoryとChatPromptTemplate

In [ ]:
#OpenAIが提供する大規模言語モデル（ChatGPTなどのGPT-4やGPT-3.5）を、LangChain上で操作するためのクラス
from langchain_openai import ChatOpenAI
#AIに過去の会話をすべて記憶させるための最も基本的な機能（クラス）をプログラムに読み込む
from langchain.memory import ConversationBufferMemory
#過去の会話のやり取り（履歴）を自動的に記憶・管理しながらAIと対話（チャット）するためのクラスをインポート
from langchain.chains import ConversationChain

#LangChainでチャットモデル用のプロンプト（テンプレート）を効率的に構築
#①プロンプト全体をまとめる親オブジェクトの設定
#②ユーザー（人間）からの入力を表現するテンプレート
#③会話履歴（List[BaseMessage]）などのメッセージのリストを後からそのまま挿入
from langchain.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
)

#システム（AI）に対する前提条件や役割指示を記述するメッセージオブジェクト
from langchain.schema import SystemMessage

system_template = """
あなたはPython初心者向けの演習問題を出題するAIアシスタントです。
データ型、条件分岐、繰り返し処理などの基本的なトピックをカバーする
シンプルな問題を作成してください。

【制約事項】
・与えられたテーマに対して、1つの演習問題のみ出題してください。
"""

#提示されたコードは、「AIの役割」「過去の会話履歴」「最新のユーザー入力」の3要素をまとめたプロンプト（指示書）の型を定義する
prompt = ChatPromptTemplate.from_messages([
    #AI（LLM）に対して前提条件や役割を与えるためのメッセージ
    SystemMessage(content=system_template),
    #会話履歴が入る動的な「置き場」を確保する
    MessagesPlaceholder(variable_name="history"),
    #ユーザー（人間）からの入力を模したメッセージのテンプレート
    HumanMessagePromptTemplate.from_template("{input}"),
])

#LangChain（PythonのLLM開発フレームワーク）でOpenAIの言語モデルを呼び出すためのインスタンス
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

#LangChain で会話の履歴（コンテキスト）をそのまま記憶・管理する
memory = ConversationBufferMemory(return_messages=True)

#ConversationChain は、LangChainで会話（対話）システムを構築するための最もシンプルな構造（Chain）を定義
#①使用する言語モデル（OpenAIやAnthropicなど）を指定します。
#②話の履歴を保存・管理するメモリ（ConversationBufferMemoryなど）を指定します。
#③実行時にプロンプトの内容や思考プロセスなどの詳細ログを画面に出力するように設定します。
chain = ConversationChain(
    llm=llm,
    prompt=prompt,
    memory=memory,
    verbose=True,
)

#チェーン（chain）に対して "条件分岐" という入力プロンプトを渡し、モデルからの推論・回答を取得する実行命令
chain.predict(input="条件分岐")

In [ ]:
chain.predict(input="繰り返し処理")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.chain import ConversationChain

from langchain.prompt import(
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    MessagePlaceholder,
)

from langchain.schema import SystemMessage

system_template = """
あなたはPython初心者向けの演習問題を出題するAIアシスタントです。
データ型、条件分岐、繰り返し処理などの基本的なトピックをカバーする
シンプルな問題を作成してください。

【制約事項】
・与えられたテーマに対して、1つの演習問題のみ出題してください。
・解答のヒントと解答例は、出題時には出さないでください。
・ユーザーから求められた場合に、解答のヒントと解答例を出す旨を出題時にのみ伝えてください。
・解答のヒントを求められた場合は、解答のヒントを出してください。
・解答例をを求められた場合は、解答例を出してください。
・Pythonのテーマが与えられた場合、毎回新しい問題を出題してください。
・ヒントや解答を求められた際は、新しい問題の出題はしないでください。
"""

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=system_template),
    MessagesPlaceholder(variable_name="history"),
    HumanMessagePromptTemplate.from_template("{input}"),
])

llm =ChatOpenAI(model_name="chat-4o-mini",temperature=0.5)

memory = ConversationBufferMemory(return_messages=True)

chain = Conversationchain(
    llm=llm,
    prompt=prompt,
    memory=memory
)




In [ ]:
count = 0
while True:
    count += 1
    if count == 1:
        input_message = "Pythonの学習テーマを入力してください（例: 条件分岐、繰り返し処理など）: "
    else:
        input_message = "ヒントは「1」、解答例は「2」、他の演習問題は「3」（「quit」で終了）: "
    input_data = input(input_message)

    if input_data.lower() == "quit":
        break

    if input_data == "1":
        input_message = "現在出題中の演習問題について、ヒントをください。"
    elif input_data == "2":
        input_message = "現在出題中の演習問題について、解答例をください。"
    elif input_data == "3":
        input_data = input("Pythonの学習テーマを入力してください（例: 条件分岐、繰り返し処理など）: ")

    print(chain.predict(input=input_data))